In [12]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder

In [13]:
train = pd.read_csv("../data/train.csv")
store = pd.read_csv("../data/store.csv")

df = train.merge(store, on="Store", how="left")

notebook_cell.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv("../data/train.csv")


In [14]:
df["Date"] = pd.to_datetime(df["Date"])

In [15]:
df["CompetitionDistance"] = df["CompetitionDistance"].fillna(df["CompetitionDistance"].median())

df["CompetitionOpenSinceMonth"] = df["CompetitionOpenSinceMonth"].fillna(0)

df["CompetitionOpenSinceYear"] = df["CompetitionOpenSinceYear"].fillna(0)

df["Promo2SinceWeek"] = df["Promo2SinceWeek"].fillna(0)

df["Promo2SinceYear"] = df["Promo2SinceYear"].fillna(0)

df["PromoInterval"] = df["PromoInterval"].fillna("None")

In [16]:
df["Year"] = df["Date"].dt.year

df["Month"] = df["Date"].dt.month

df["Day"] = df["Date"].dt.day

df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)

df["DayOfWeek"] = df["Date"].dt.dayofweek

df["Quarter"] = df["Date"].dt.quarter

df["IsWeekend"] = df["DayOfWeek"].isin([5,6]).astype(int)

df["IsMonthStart"] = df["Date"].dt.is_month_start.astype(int)

df["IsMonthEnd"] = df["Date"].dt.is_month_end.astype(int)

In [17]:
competition_date = pd.to_datetime(
    dict(
        year=df["CompetitionOpenSinceYear"].replace(0, 1900),
        month=df["CompetitionOpenSinceMonth"].replace(0, 1),
        day=1
    )
)

df["CompetitionAgeMonths"] = (
    (df["Date"] - competition_date)
    .dt.days
    .clip(lower=0) // 30
)

In [18]:
promo_date = pd.to_datetime(
    dict(
        year=df["Promo2SinceYear"].replace(0, 1900),
        month=1,
        day=1
    )
)

df["PromoAgeMonths"] = (
    (df["Date"] - promo_date)
    .dt.days
    .clip(lower=0) // 30
)

In [19]:
encoder = LabelEncoder()

categorical_cols = [
    "StoreType",
    "Assortment",
    "PromoInterval",
    "StateHoliday"
]

for col in categorical_cols:
    df[col] = encoder.fit_transform(df[col].astype(str))

In [20]:
df = df[df["Open"] == 1]

In [21]:
df = df[df["Sales"] > 0]

In [22]:
df.to_csv(
    "../data/processed_train.csv",
    index=False
)